# Testing del Clasificador de Palabras Clave (Importado)
Este notebook evalúa el rendimiento del algoritmo importando directamente la clase desde `classifier.py` y leyendo las reglas desde `config.json`.

In [ ]:
import pandas as pd
# Importamos la clase directamente desde tu archivo local
from classifier import KeywordClassifier

In [ ]:
# 1. Instanciar el clasificador usando el config.json de la raíz
clasificador = KeywordClassifier.from_json("config.json")
print("Clasificador inicializado correctamente desde config.json.")

In [ ]:
# 2. Crear un dataset de prueba con la Verdad Fundamental (Ground Truth)
datos_prueba = [
    {"texto": "La vpn me da un error al conectar", "categoria_real": "Soporte TI"},
    {"texto": "Quiero saber el precio de las licencias", "categoria_real": "Ventas y Comercial"},
    {"texto": "Me hicieron un cobro doble en la tarjeta", "categoria_real": "Facturación y Pagos"},
    {"texto": "El servicio es pésimo, iré al sernac", "categoria_real": "Reclamos"},
    {"texto": "Hola buenos días", "categoria_real": "Sin clasificación"},
    {"texto": "Necesito una cotización urgente", "categoria_real": "Ventas y Comercial"},
    {"texto": "El sistEma tiene caídas intermitentes", "categoria_real": "Soporte TI"},
    # Casos insertados a propósito para probar errores (Falsos negativos)
    {"texto": "Tengo un problema técnico en mi equipo", "categoria_real": "Soporte TI"},
    {"texto": "Quiero adquirir más productos", "categoria_real": "Ventas y Comercial"}
]
df = pd.DataFrame(datos_prueba)
df.head()

In [ ]:
# 3. Aplicar el clasificador al DataFrame
def aplicar_clasificador(texto):
    res = clasificador.classify_text(texto)
    return pd.Series([res['categoria'], res['keyword_match']])

df[['categoria_predicha', 'evidencia_keyword']] = df['texto'].apply(aplicar_clasificador)
display(df)

In [ ]:
# 4. Evaluar el rendimiento (Métricas)
df['acierto'] = df['categoria_real'] == df['categoria_predicha']
accuracy = df['acierto'].mean()
print(f"\nPrecisión Global (Accuracy): {accuracy * 100:.2f}%")

# Filtrar y mostrar dónde falló el algoritmo
errores = df[~df['acierto']]
if not errores.empty:
    print("\n--- CASOS FALLIDOS (Requieren agregar keywords al JSON) ---")
    display(errores)
else:
    print("\n¡El clasificador acertó el 100% de los casos!")